In [11]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)
from sklearn.model_selection import train_test_split
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import load_dataset, Dataset

import transformers
from packaging import version


In [4]:
RANDOM_SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3
MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "./results_test"
FINAL_MODEL_DIR = "./final_model_test"
DATASET_NAME = "ealvaradob/phishing-dataset"
CONFIG_NAME = "combined_reduced"

# Создание необходимых директорий
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## ЗАГРУЗКА ДАТАСЕТА

In [16]:

print("=" * 60)
print("ЗАГРУЗКА ДАТАСЕТА")
print("=" * 60)

# Загружаем датасет с HuggingFace
import requests

def load_phishing_dataset_from_json():
    """Загружает датасет из JSON-файла на Hugging Face"""
    # уже загружен
    # json_url = "https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/combined_reduced.json"
    
    # print("Загрузка JSON-файла (это может занять несколько минут)...")
    
    # # Скачиваем JSON
    # response = requests.get(json_url, stream=True)
    # response.raise_for_status()
    
    # # Сохраняем временно на диск
    temp_file = "combined_reduced.json"
    # with open(temp_file, 'wb') as f:
    #     for chunk in response.iter_content(chunk_size=8192):
    #         f.write(chunk)
    
    print("JSON загружен, читаем данные...")
    
    # Читаем JSON (ожидается формат: список объектов с ключами text и label)
    data = []
    with open(temp_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # os.remove(temp_file)
    df = pd.DataFrame(data)
    # Создаем DataFrame
    # Загружаем данные
    print(f"\nЗагружено {len(df)} примеров")
    print(f"Колонки: {df.columns.tolist()}")
    print(f"Распределение классов:\n{df['label'].value_counts()}")
    
    # Если колонка называется не "text" а "message" или что-то ещё
    text_column = None
    for col in ['text', 'message', 'content', 'body']:
        if col in df.columns:
            text_column = col
            break
    
    if text_column and text_column != 'message':
        df = df.rename(columns={text_column: 'message'})
    
    return df

# Загружаем данные
df = load_phishing_dataset_from_json()
print("\nПервые 5 записей:")
print(df.head())

ЗАГРУЗКА ДАТАСЕТА
JSON загружен, читаем данные...

Загружено 77677 примеров
Колонки: ['text', 'label']
Распределение классов:
label
0    44975
1    32702
Name: count, dtype: int64

Первые 5 записей:
                                             message  label
0  <!doctypehtml><html lang=en xml:lang=en xmlns=...      0
1                  http://online0mgeving.ga/triodos/      1
2  metronews.ca/webapp/Login.aspx?logout=true&rur...      0
3      https://rarkuntem.co.jp.fpjiehk.cn/index1.php      1
4                        freebase.com/view/m/0ct05qn      0


In [ ]:


if "train" in dataset:
    # У датасета есть разделение train/test
    train_data = dataset["train"]
    if "test" in dataset:
        test_data = dataset["test"]
    else:
        test_data = None
else:
    train_data = dataset
    test_data = None

# Преобразуем в pandas для удобства анализа и разделения
if isinstance(train_data, Dataset):
    train_df = train_data.to_pandas()
else:
    train_df = train_data

if test_data is not None and isinstance(test_data, Dataset):
    test_df = test_data.to_pandas()
else:
    test_df = None

if test_df is None:
    print("Датасет не содержит тестовой выборки. Выполняем разделение самостоятельно.")
    text_col = "text" if "text" in train_df.columns else train_df.columns[0]
    label_col = "label" if "label" in train_df.columns else train_df.columns[1]
    print(f"Используем колонку текста: '{text_col}', метки: '{label_col}'")

    train_df = train_df.rename(columns={text_col: "message", label_col: "label"})

    temp_df, test_df = train_test_split(
        train_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=train_df["label"],
    )

    val_ratio = VAL_SIZE / (1 - TEST_SIZE)
    train_df, val_df = train_test_split(
        temp_df,
        test_size=val_ratio,
        random_state=RANDOM_SEED,
        stratify=temp_df["label"],
    )
    print(f"Размеры выборок: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
else:
   
    text_col = "text" if "text" in train_df.columns else train_df.columns[0]
    label_col = "label" if "label" in train_df.columns else train_df.columns[1]
    train_df = train_df.rename(columns={text_col: "message", label_col: "label"})
    test_df = test_df.rename(columns={text_col: "message", label_col: "label"})

    train_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=RANDOM_SEED,
        stratify=train_df["label"],
    )
    print(f"Исходный train разбит: train={len(train_df)}, val={len(val_df)}")
    print(f"Тест: {len(test_df)}")

# Вывод распределения классов
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    spam_ratio = df["label"].mean() * 100
    print(f"{name}: всего {len(df)}, спам/фишинг: {df['label'].sum()} ({spam_ratio:.1f}%)")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ealvaradob/phishing-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


ЗАГРУЗКА ДАТАСЕТА


RuntimeError: Dataset scripts are no longer supported, but found phishing-dataset.py

## ЗАГРУЗКА ТОКЕНИЗАТОРА И ТОКЕНИЗАЦИЯ

In [ ]:
print("\n" + "=" * 60)
print("ЗАГРУЗКА ТОКЕНИЗАТОРА И ТОКЕНИЗАЦИЯ")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    """Токенизирует тексты и добавляет метки."""
    tokenized = tokenizer(
        examples["message"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    tokenized["labels"] = examples["label"]
    return tokenized

# Создаем объекты Dataset из pandas DataFrame
train_dataset = Dataset.from_pandas(train_df[["message", "label"]])
val_dataset = Dataset.from_pandas(val_df[["message", "label"]])
test_dataset = Dataset.from_pandas(test_df[["message", "label"]])

# Токенизация с удалением исходного текста
tokenized_train = train_dataset.map(tokenize_function, batched=True, batch_size=16)
tokenized_val = val_dataset.map(tokenize_function, batched=True, batch_size=16)
tokenized_test = test_dataset.map(tokenize_function, batched=True, batch_size=16)

tokenized_train = tokenized_train.remove_columns(["message"])
tokenized_val = tokenized_val.remove_columns(["message"])
tokenized_test = tokenized_test.remove_columns(["message"])

print("Токенизация завершена.")

## ЗАГРУЗКА МОДЕЛИ

### ЗАГРУЗКА МОДЕЛИ BERT

In [3]:
print("\n" + "=" * 60)
print("ЗАГРУЗКА МОДЕЛИ BERT")
print("=" * 60)

num_labels = len(train_df["label"].unique())
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)
print(f"Модель загружена. Количество классов: {num_labels}")



ЗАГРУЗКА МОДЕЛИ BERT


NameError: name 'train_df' is not defined

In [ ]:
def compute_metrics(eval_pred):
    """Вычисляет accuracy, precision, recall, f1."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary" if num_labels == 2 else "weighted"
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

## Параметры обучения

In [ ]:
import transformers
from packaging import version

print(f"Используется transformers версии: {transformers.__version__}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch", 
    save_strategy="no",
    report_to=[],
    seed=RANDOM_SEED,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Обучение

In [ ]:
print("\n" + "=" * 60)
print("ОБУЧЕНИЕ МОДЕЛИ")
print("=" * 60)

try:
    trainer.train()
    print("Обучение завершено успешно!")
except Exception as e:
    print(f"Ошибка во время обучения: {e}")
    import traceback
    traceback.print_exc()
    exit(1)

# Сохранение модели и токенизатора
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Модель сохранена в {FINAL_MODEL_DIR}")

In [ ]:
print("\n" + "=" * 60)
print("ОЦЕНКА НА ТЕСТОВЫХ ДАННЫХ")
print("=" * 60)

test_results = trainer.predict(tokenized_test)
print("Метрики на тесте:")
for key, value in test_results.metrics.items():
    print(f"  {key}: {value:.4f}")

y_true = test_results.label_ids
y_pred = np.argmax(test_results.predictions, axis=1)
# Вероятности для ROC
if num_labels == 2:
    y_scores = torch.softmax(torch.tensor(test_results.predictions), dim=1).numpy()[:, 1]
else:
    y_scores = None